# 06 — Real Hardware, Data Saving & REST Service

This notebook covers the three things you do differently on real hardware vs simulation:

1. Connect to the QICK board via Pyro4
2. Populate `system_cfg.py` with your lab's channel map
3. Save / reload experimental data (HDF5)
4. Start the REST service for remote experiment submission

> **Note:** cells in section 1 require a live QICK board and will not run in simulation.

## 1. Connecting to the QICK board

`QICKBackend` wraps the Pyro4 proxy and provides the same interface as `SimulatedBackend`.

In [ ]:
# ── HARDWARE ONLY — edit IP and port to match your board ──
import sys; sys.path.insert(0, '../')

QICK_IP   = '192.168.1.100'   # ← change this
QICK_PORT = 8888              # ← change this

from reconstruct import QICKBackend, BaseExperiment

# This call blocks until the Pyro4 proxy connects
backend = QICKBackend.from_pyro4(QICK_IP, QICK_PORT)

# activate() sets BaseExperiment._soc / _soccfg globally
# and also sets the data path
DATA_PATH = r'D:/data/my_experiment'
backend.activate(data_path=DATA_PATH)

print('Connected:', backend)

## 2. Hardware config — edit `system_cfg.py`

Open `reconstruct/config/system_cfg.py` and fill in your lab's channel assignments
and starting frequencies.  The file contains a `config_list` (list of dicts, one per qubit)
and a `DATA_PATH` string.

```python
# reconstruct/config/system_cfg.py  (excerpt)
DATA_PATH = r'D:/data/my_experiment'

config_list = [
    {
        'name': 'Q1',
        'ch':  {'ro_ch': 0, 'res_ch': 0, 'qb_ch': 1},
        'res': {'res_freq_ge': 6701.2, 'res_gain': 0.5, 'res_length': 2.0},
        'qb':  {'qb_freq_ge': 4998.7, 'pi_gain_ge': 0.512, 'sigma': 0.025},
        'reps': 1000, 'relax_delay': 300.0, 'steps': 201,
    },
]
```

Once set, load it:

In [ ]:
from reconstruct import ExperimentConfig
from reconstruct.config.system_cfg import config_list   # your lab config

cfg_all = ExperimentConfig(config_list)
cfg = cfg_all.get_qubit('Q1')
print('Loaded qubits:', cfg_all.qubit_names())

## 3. Run an experiment and save data

`ExperimentData.save(path)` writes an HDF5 file with raw IQ, axes, fit params, and the
full config snapshot.  `ExperimentData.load(path)` reloads it.

In [ ]:
# Works with both real hardware and SimulatedBackend
import sys; sys.path.insert(0, '../')
import tempfile, os

from reconstruct import SimulatedBackend, ExperimentConfig
from reconstruct.experiments.resonator import ResonatorSpec

# Switch to simulated if no hardware
sim_backend = SimulatedBackend()
sim_backend.activate()

raw_config = [{
    "name": "Q1",
    "ch":  {"ro_ch": 0, "res_ch": 0, "qb_ch": 1},
    "res": {"res_freq_ge": 6700.0, "res_gain": 0.5, "res_length": 2.0},
    "qb":  {"qb_freq_ge": 5000.0, "pi_gain_ge": 0.5, "sigma": 0.025},
    "reps": 100, "relax_delay": 300.0, "steps": 51, "kappa": 4.0,
}]
cfg_all = ExperimentConfig(raw_config)
cfg = cfg_all.get_qubit('Q1')

result = ResonatorSpec(cfg, backend=sim_backend).run(py_avg=5)
print('Experiment ID:', result.experiment_id)

In [ ]:
# Save
save_path = os.path.join(tempfile.gettempdir(), f'res_spec_{result.experiment_id}.h5')
result.save(save_path)
print('Saved to:', save_path)

# Reload
reloaded = ExperimentData.load(save_path)
print('Reloaded experiment_type:', reloaded.experiment_type)
print('Reloaded scalar_result  :', reloaded.scalar_result)

In [ ]:
from reconstruct import ExperimentData

# Labber-format save (compatible with Labber data viewer)
# expt.saveLabber()   # uses DATA_PATH from system_cfg automatically

# JSON-serialisable dict (for logging, REST responses, etc.)
d = result.to_dict()
print('Keys in to_dict():', list(d.keys()))

## 4. REST Service

The FastAPI service lets you submit experiments remotely — useful for running the
calibration framework from a control PC while the QICK board is in the lab rack.

**Start the server** (run in a terminal, not this notebook):
```bash
cd SQC_soc
uvicorn reconstruct.service.api:app --host 0.0.0.0 --port 8000
```

Or start it programmatically:

In [ ]:
# Programmatic startup (non-blocking, runs in background thread)
import threading
import uvicorn

from reconstruct.service import create_app
from reconstruct import CalibrationStore
import tempfile, os

store = CalibrationStore(os.path.join(tempfile.gettempdir(), 'service_demo.json'))
app = create_app(cal_store=store, config_all=cfg_all, backend=sim_backend)

server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning'))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
print('Service running at http://127.0.0.1:8000')

In [ ]:
# Submit a job via HTTP
import requests, time, json

BASE = 'http://127.0.0.1:8000'

payload = {
    'experiment': 'ResonatorSpec',
    'qubit': 'Q1',
    'py_avg': 3,
}

r = requests.post(f'{BASE}/experiments/run', json=payload)
job_id = r.json()['job_id']
print('Job submitted:', job_id)

# Poll until done
for _ in range(20):
    status = requests.get(f'{BASE}/experiments/{job_id}/status').json()
    if status['status'] in ('done', 'error'):
        break
    time.sleep(0.5)

print('Status:', status)

if status['status'] == 'done':
    result_json = requests.get(f'{BASE}/experiments/{job_id}/result').json()
    print('scalar_result:', result_json.get('scalar_result'))

In [ ]:
# API endpoints reference
endpoints = requests.get(f'{BASE}/openapi.json').json()
for path in endpoints['paths']:
    print(path)

## Hardware validation checklist

Run these steps in order on first hardware connection:

1. `python -c "import reconstruct; print(reconstruct.__version__)"` — package loads
2. `QICKBackend.from_pyro4(IP, PORT)` — Pyro4 connection succeeds
3. `ResonatorSpec` — verify resonator dip appears at the expected frequency
4. `QubitSpec` — verify qubit peak appears
5. `PowerRabi` — verify clean Rabi oscillation
6. `Ramsey` — verify detuning < 1 MHz after QubitSpec
7. `AutoCalibrate.run(skip=('spin_echo', 't1', 'ss_opt'))` — fast first-pass calibration
8. `AutoCalibrate.run()` — full calibration